# AtlasOps — SFT Training on Kaggle (Free GPU T4 x2)

This notebook runs 4-bit NF4 QLoRA Supervised Fine-Tuning on `Qwen/Qwen2.5-7B-Instruct` using the verified AtlasOps multi-agent incident trajectory dataset (`artifacts/evidence/stage7/sft_corpus_manifest.json`).

### Prerequisites
- In Kaggle Notebook Settings, set **Accelerator** to **GPU T4 x2** or **GPU P100**.
- Set **Internet** to **On**.

In [ ]:
# Step 1: Clone the canonical AtlasOps repository
!git clone https://github.com/virajchoudhary/AtlasOps.git /kaggle/working/AtlasOps
%cd /kaggle/working/AtlasOps
!git status

In [ ]:
# Step 2: Install required training packages
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# Step 3: Inspect GPU availability and dataset integrity
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")

import json
from pathlib import Path
manifest_path = Path("artifacts/evidence/stage7/sft_corpus_manifest.json")
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"Corpus SHA-256: {manifest.get('canonical_lf_sha256')}")
    print(f"Total Examples: {manifest.get('total_examples')}")

In [ ]:
# Step 4: Run SFT QLoRA Training Pipeline
!python -m training.sft \
    --model "Qwen/Qwen2.5-7B-Instruct" \
    --data "data/sft_corpus_train.jsonl" \
    --output "/kaggle/working/qwen2.5-7b-atlasops-sft" \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 4 \
    --lr 2e-4

In [ ]:
# Step 5: Verify Saved Adapter Checkpoint
import os
output_dir = "/kaggle/working/qwen2.5-7b-atlasops-sft"
if os.path.exists(output_dir):
    print(f"Saved checkpoint files in {output_dir}:")
    for f in os.listdir(output_dir):
        print(f" - {f}")